In [ ]:
from main import*
from circle_utilities import*

from IPython.display import clear_output # To clear tqdm bars

DTM parameters

In [ ]:
ms = [0.05, 0.1, 0.2]
betas = [0.5, 1, 2]
p = 2

n_m, n_beta = len(ms), len(betas)
params = np.array([[DTM_arg(m, p, beta) for beta in betas] for m in ms])

precision_segment = 20

Identify the geodesic : run the FDTM algorithm on a uniform circle grid to detect how many segments make the geodesic.

In [ ]:
N = 500

# Endpoints
theta_x, theta_y = -1/12, -11/12
x = np.array([np.cos(np.pi*theta_x), np.sin(np.pi*theta_x)])
y = np.array([np.cos(np.pi*theta_y), np.sin(np.pi*theta_y)])

# Uniform point cloud on the circle
X = circle_grid(N)

# Get indices of closest point in the grid to x and y
i, j = np.argmin(norm(X - x, axis=1)), np.argmin(norm(X - y, axis=1))

dtms = np.vectorize(lambda d : DTM(X, d, dtm_circle))(params)
paths = np.zeros_like(dtms)
Ms = np.zeros_like(dtms)

for k in tqdm(range(n_m)):
    for l in tqdm(range(n_beta), leave=False):
        dtm = dtms[k, l]
        Ms[k, l] = FDTM(X, dtms[k, l]).select_edges().make_edges(precision_segment=precision_segment).shortest_path(source=i)
        paths[k, l] = Ms[k, l].path((i, j))[0]

clear_output()

In [ ]:
geodesic_lengths = np.zeros_like(dtms)  # Amount of points on the geodesics for each set of parameters
geodesic = np.zeros_like(dtms)  # Truc geodesic points for each set of parameters

for k in tqdm(range(n_m)):
    for l in tqdm(range(n_beta), leave=False):
        geodesic_length = len(paths[k][l])
        segments_lengths = norm(X[paths[k][l][1:]] - X[paths[k][l][:-1]], axis=1)
        geodesic_lengths[k, l] = 1 + np.sum(segments_lengths > segments_lengths.max() * 0.8)  # Ignore small edges that would result of approximation error

        if geodesic_lengths[k, l] != geodesic_length:
            print(f'Segment lengths: {segments_lengths}')
            print('Warning : not counting smaller segments')

        assert abs(theta_x-theta_y) <= 1 and N > 10*geodesic_lengths[k, l]  # Arguments should be at most 1 apart and the amount of points on the grid should be large enough compare to the number of points on the geodesic
        theta_geodesic = np.linspace(theta_x, theta_y, geodesic_lengths[k, l])  # Actual intermediate points (angles) of the true geodesic
        geodesic[k, l] = np.stack((np.cos(np.pi*theta_geodesic), np.sin(np.pi*theta_geodesic)), axis=1)

clear_output()

Compute the FDTM of the geodesics by using only the points from the geodesics as a point cloud

In [ ]:
Ms = np.zeros_like(dtms)

for k in tqdm(range(n_m)):
    for l in tqdm(range(n_beta), leave=False):
        Ms[k, l] = FDTM(geodesic[k, l], dtms[k, l]).make_edges(precision_segment=precision_segment).shortest_path().make_path((0, geodesic_lengths[k, l]-1))

clear_output()

Plot DTM surface and geodesics

In [ ]:
N_surface = 300
grid = square_grid(N_surface, bounding_box=1.1)

fig_size = 5
fig, ax = plt.subplots(n_m, n_beta, figsize=(5*n_beta, 5*n_m))

for k in range(n_m):
    for l in range(n_beta):
        vmax = 1
        if dtms[k, l].beta >= 2 : vmax = 0.5  # Hardcoded choice
        plot_2d(Ms[k, l], grid, (0, geodesic_lengths[k, l]-1), ax=ax[k, l], path_color='yellowgreen', marker='.', markersize=10, linewidth=3, vmin=0, vmax=vmax)

        # Plot Support circle
        ax[k, l].add_patch(mpl.patches.Circle((0,0), 1, color='black', fill=False, label='Support', linewidth=1.5))

        # Colorbar
        mesh = ax[k, l].images[-1]
        fig.colorbar(mesh, ax=ax[k, l], label='DTM', fraction=0.045)  # `fraction` is hardcoded and adjusts the size of the colorbar

plt.show()

Save figures

In [ ]:
save = True  # Save figures

In [ ]:
N_surface = 300
grid = square_grid(N_surface, bounding_box=1.1)

for k in range(n_m):
    for l in range(n_beta):
        fig, ax = plt.subplots()
        
        vmax = 1
        if dtms[k, l].beta >= 2 : vmax = 0.5  # Hardcoded choice
        plot_2d(Ms[k, l], grid, (0, geodesic_lengths[k, l]-1), ax=ax, path_color='yellowgreen', marker='.', markersize=10, linewidth=3, vmin=0, vmax=vmax)

        # Plot Support circle
        ax.add_patch(mpl.patches.Circle((0,0), 1, color='black', fill=False, label='Support', linewidth=1.5))

        # Colorbar
        mesh = ax.images[-1]
        fig.colorbar(mesh, ax=ax, label='DTM', fraction=0.045)  # `fraction` is hardcoded and adjusts the size of the colorbar

        if save : fig.savefig(f'figures\\circle geodesics\\m_{dtms[k, l].m}_beta_{dtms[k, l].beta}.png', bbox_inches='tight', transparent=True, dpi=200)
        plt.close()